# Benchmark: how big a problem this PC can train

How long the fixed RL layer (`core/fixed/`) takes per environment step at the
market's problem size, and, with the market's own cost per step from the clean
branch, how many hours a training run of a given length would take here.

**The environment.** `MarketShapedEnv` (`core/envs/shapes.py`): each agent
observes 42 numbers and sends 4 actions, as a MARLEM market agent; episodes of 24
steps (one day, hourly). The observations are random and the reward is cheap, so
the time measured is RLlib and the networks alone. Grouped paradigms see all
agents at once:

| | DTDE (one module per agent) | CTCE and CTDE (one module for the group) |
|---|---|---|
| Module input | 42 | 42 · N |
| Module output | 4 | 4 · N |

**What one benchmark measures** (`core/fixed/benchmark.py`):

1. **Build**: the networks, optimisers and sampler.
2. **Warm-up** (not timed): iterations until the learner has made its first
   update. PPO: one batch. SAC: `num_steps_sampled_before_learning_starts`
   environment steps, lowered here so learning starts almost at once. APPO: until
   its learner first reports, which is late (it reports only now and then).
3. **Timed**: whole training iterations (sample, then learn) until
   `TIMED_ENV_STEPS` environment steps: seconds per 1000 environment steps and
   gradient steps.
4. **Sampling alone**: the same sampler acting with the current policy, no
   learning. The difference to 3 is the learning.

Each benchmark runs alone, in a fresh process (no other runs sharing the CPU, as
the probe runs did). Results go to `runs/benchmark/` (one JSON per benchmark),
and a finished one is not run again unless `RETIME = True`.

In [1]:
import json
import multiprocessing
import os
import sys
import time
from concurrent.futures import ProcessPoolExecutor
from pathlib import Path

# RL root; imports use its parent so market core/ stays separate.
ROOT = next(p / "RL" for p in [Path.cwd(), *Path.cwd().parents] if (p / "RL" / "core").is_dir())
sys.path.insert(0, str(ROOT.parent))
os.environ["PYTHONPATH"] = os.pathsep.join(filter(None, [str(ROOT.parent), os.environ.get("PYTHONPATH", "")]))  # the benchmark processes import RL.core too

import numpy as np
import pandas as pd
import torch

from RL.core.envs.shapes import MarketShapedEnv
from RL.core.fixed.benchmark import Bench, benchmark

pd.set_option("display.precision", 2)
pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 30)
print("torch", torch.__version__, "| GPU:",
      torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none",
      "| CPU threads:", torch.get_num_threads())

torch 2.6.0+cu126 | GPU: NVIDIA GeForce RTX 3060 Ti | CPU threads: 6


## a. Choose what to benchmark

- `ALGORITHMS`, `PARADIGMS`, `AGENTS` (N), `DEVICES` (`"cpu"`, `"gpu"`: where the
  learner trains; sampling is always on the CPU).
- `PRESETS`: named settings per algorithm. Every key is passed to the algorithm
  as it is; section b shows what each one means in gradient steps.

**Why the PPO batch size is fixed at 1000 here.** PPO's cost per environment step
does not depend on the batch size: every sampled step is trained on
`num_epochs` times, in minibatches of `minibatch_size`, so a batch of B steps
costs `num_epochs · B / minibatch_size` gradient steps, and per 1000 environment
steps that is `num_epochs · 1000 / minibatch_size` for any B. A smaller batch
only makes each timed iteration shorter. The two presets differ where the cost
is: `"paper"` keeps the paper's minibatch 128 and 30 epochs (about 234 gradient
steps per 1000 environment steps), `"light"` uses minibatch 512 and 10 epochs
(about 20).

**SAC and APPO** are here for comparison; the next experiments use PPO. SAC's
`"quick"` preset starts learning after 256 steps (its default is 1500) and keeps
RLlib's default replay ratio (about one gradient step of 256 samples per
environment step).

In [2]:
ALGORITHMS = ["ppo"]
PARADIGMS = ["ctce", "ctde", "dtde"]
AGENTS = [5, 13, 39, 118]
DEVICES = ["cpu", "gpu"]
PARTNER_PREFERENCE = False  # True: 42 + N observations per agent
EPISODE_LENGTH = 24
TIMED_ENV_STEPS = 1000  # at least one whole iteration is timed
SAMPLE_ENV_STEPS = 500
MAX_SECONDS = 600
RETIME = False  # True: run finished benchmarks again

PRESETS = {
    "ppo": {
        "paper": {"train_batch_size_per_learner": 1000, "minibatch_size": 128,
                  "num_epochs": 30},
        "light": {"train_batch_size_per_learner": 1000, "minibatch_size": 512,
                  "num_epochs": 10},
    },
    "appo": {
        "default": {"train_batch_size_per_learner": 500},
    },
    "sac": {
        "quick": {"num_steps_sampled_before_learning_starts": 256},
    },
}
OUT = ROOT / "runs" / "benchmark"
MARKET_STEP_CSV = ROOT.parent / "notebooks" / "results" / "market_step.csv"

## b. What each preset means

Computed from the settings, before running anything: steps the learner waits for
before its first update, and gradient steps per 1000 environment steps per
module (DTDE has N modules, each taking these steps on its own agent's data).
SAC's and APPO's gradient steps depend on how RLlib paces them; they are measured
in section d.

In [3]:
def preset_table() -> pd.DataFrame:
    rows = []
    for algorithm in ALGORITHMS:
        for name, s in PRESETS[algorithm].items():
            row = {"algorithm": algorithm, "preset": name} | s
            if algorithm == "ppo":
                row["warm-up env steps"] = s["train_batch_size_per_learner"]
                row["gradient steps / 1k env steps"] = (
                    s["num_epochs"] * 1000 / s["minibatch_size"]
                )
            elif algorithm == "sac":
                row["warm-up env steps"] = s["num_steps_sampled_before_learning_starts"]
            rows.append(row)
    return pd.DataFrame(rows).set_index(["algorithm", "preset"])


preset_table()

train_batch_size_per_learner  minibatch_size  num_epochs  warm-up env steps  gradient steps / 1k env steps
algorithm preset                                                                                                            
ppo       paper                           1000             128          30               1000                         234.38
          light                           1000             512          10               1000                          19.53

## c. Run

Smallest problems first, so the table fills from the cheap end. The time of each
benchmark is printed as it finishes; the larger DTDE cases take several minutes
each (one module, one optimiser and one update per agent).

In [4]:
def name(algorithm, paradigm, n, preset, device) -> str:
    return f"{algorithm}_{paradigm}_n{n}_{preset}_{device}"


def plan() -> list[Bench]:
    benches = []
    for n in AGENTS:
        for algorithm in ALGORITHMS:
            for preset, training in PRESETS[algorithm].items():
                for paradigm in PARADIGMS:
                    for device in DEVICES:
                        benches.append(Bench(
                            algorithm,
                            paradigm,
                            MarketShapedEnv,
                            {"n_agents": n, "episode_length": EPISODE_LENGTH,
                             "partner_preference": PARTNER_PREFERENCE},
                            training=training,
                            num_gpus=int(device == "gpu"),
                            timed_env_steps=TIMED_ENV_STEPS,
                            sample_env_steps=SAMPLE_ENV_STEPS,
                            max_seconds=MAX_SECONDS,
                            out=OUT / f"{name(algorithm, paradigm, n, preset, device)}.json",
                        ))
    return benches


def finished(bench: Bench) -> bool:
    path = Path(bench.out)
    return path.exists() and json.loads(path.read_text())["status"] == "completed"


def run_all(benches: list[Bench]) -> None:
    pending = [b for b in benches if RETIME or not finished(b)]
    print(f"{len(benches) - len(pending)} finished, {len(pending)} to run")
    context = multiprocessing.get_context("spawn")
    for bench in pending:
        start = time.time()
        # A fresh process per benchmark: nothing left over from the one before.
        with ProcessPoolExecutor(1, mp_context=context, max_tasks_per_child=1) as pool:
            try:
                record = pool.submit(benchmark, bench).result()
                note = record["status"]
                if record["status"] == "completed":
                    note = f"{record['results']['s_per_1k_env_steps']:7.1f} s / 1k steps"
                else:
                    note += " " + record.get("error", "")[:120]
            except Exception as error:
                note = f"process failed: {error!r}"[:160]
        print(f"{Path(bench.out).stem:32s} {time.time() - start:6.0f} s   {note}",
              flush=True)


BENCHES = plan()
run_all(BENCHES)

0 finished, 48 to run


ppo_ctce_n5_paper_cpu                21 s       3.3 s / 1k steps


ppo_ctce_n5_paper_gpu                22 s       4.0 s / 1k steps


ppo_ctde_n5_paper_cpu                25 s       5.2 s / 1k steps


ppo_ctde_n5_paper_gpu                28 s       7.0 s / 1k steps


...

ppo_ctce_n118_light_gpu              28 s       4.9 s / 1k steps


ppo_ctde_n118_light_cpu             119 s      46.9 s / 1k steps


ppo_ctde_n118_light_gpu              75 s      24.5 s / 1k steps


ppo_dtde_n118_light_cpu             198 s      77.3 s / 1k steps


ppo_dtde_n118_light_gpu             223 s      90.4 s / 1k steps


## d. Results

One row per benchmark. Seconds per 1000 environment steps, split into sampling
(acting, including RLlib's per-step overhead) and learning; gradient steps
counted from the samples the learner trained on; memory of the process and, on
the GPU, the learner's peak.

In [5]:
def load() -> pd.DataFrame:
    rows = []
    for bench in BENCHES:
        path = Path(bench.out)
        if not path.exists():
            continue
        r = json.loads(path.read_text())
        row = {
            "algorithm": r["algorithm"],
            "preset": path.stem.split("_")[3],
            "paradigm": r["paradigm"],
            "N": r["n_agents"],
            "device": r["device"],
            "status": r["status"],
        }
        if r["status"] == "completed":
            res = r["results"]
            row |= {
                "modules": r["n_modules"],
                "module in": r["module_obs_dim"],
                "module out": r["module_act_dim"],
                "parameters (M)": r["parameters"] / 1e6,
                "build s": r["build_s"],
                "warm-up steps": r["warmup"]["env_steps"],
                "warm-up s": r["warmup"]["seconds"],
                "s / 1k steps": res["s_per_1k_env_steps"],
                "sampling s / 1k": res["sample_s_per_1k"],
                "learning s / 1k": res["learn_s_per_1k"],
                "grad steps / s": res["gradient_steps_per_s"],
                "grad steps / 1k steps": res["gradient_steps_per_1k_env_steps"],
                "h / 1M steps": res["h_per_1M_env_steps"],
                "RAM GB": res["rss_gb"],
                "GPU GB": res["gpu_peak_gb"],
            }
        else:
            row["error"] = r.get("error", "")
        rows.append(row)
    return pd.DataFrame(rows)


results = load()
results

,algorithm,preset,paradigm,N,device,status,modules,module in,module out,parameters (M),build s,warm-up steps,warm-up s,s / 1k steps,sampling s / 1k,learning s / 1k,grad steps / s,grad steps / 1k steps,h / 1M steps,RAM GB,GPU GB
0,ppo,paper,ctce,5,cpu,completed,1,210,20,0.25,1.44,1000,3.63,3.30,1.49,1.81,74.16,245.0,0.92,0.75,0.00
1,ppo,paper,ctce,5,gpu,completed,1,210,20,0.25,1.47,1000,4.34,4.03,1.22,2.81,60.86,245.0,1.12,1.62,0.08
2,ppo,paper,ctde,5,cpu,completed,1,210,20,0.51,1.34,1000,5.31,5.16,1.65,3.51,47.45,245.0,1.43,0.78,0.00
3,ppo,paper,ctde,5,gpu,completed,1,210,20,0.51,1.50,1000,7.22,6.97,1.57,5.39,35.17,245.0,1.94,1.65,0.08
4,ppo,paper,dtde,5,cpu,completed,5,42,4,0.78,1.37,1000,9.64,9.58,2.98,6.60,127.93,1225.0,2.66,0.80,0.00
5,ppo,paper,dtde,5,gpu,completed,5,42,4,0.78,1.50,1000,15.41,16.01,2.97,13.05,76.50,1225.0,4.45,1.63,0.09
6,ppo,light,ctce,5,cpu,completed,1,210,20,0.25,1.31,1000,1.47,1.51,1.21,0.30,13.94,21.0,0.42,0.77,0.00
7,ppo,light,ctce,5,gpu,completed,1,210,20,0.25,1.43,1000,1.79,1.52,1.18,0.34,13.83,21.0,0.42,1.62,0.08
8,ppo,light,ctde,5,cpu,completed,1,210,20,0.51,1.33,1000,2.33,2.15,1.60,0.55,9.79,21.0,0.60,0.80,0.00
9,ppo,light,ctde,5,gpu,completed,1,210,20,0.51,1.43,1000,2.42,2.27,1.58,0.69,9.26,21.0,0.63,1.63,0.09


### CPU or GPU, per problem

Hours per million environment steps of the RL layer alone; the faster device per
row is the one to train on.

In [6]:
done = results[results.status == "completed"]
by_device = done.pivot_table(
    index=["algorithm", "preset", "paradigm", "N"], columns="device",
    values="h / 1M steps",
)
if {"cpu", "gpu"} <= set(by_device.columns):
    by_device["faster"] = np.where(by_device.gpu < by_device.cpu, "gpu", "cpu")
by_device

device                           cpu    gpu faster
algorithm preset paradigm N                       
ppo       light  ctce     5     0.42   0.42    cpu
                          13    0.51   0.51    cpu
                          39    0.70   0.73    cpu
                          118   1.51   1.36    gpu
                 ctde     5     0.60   0.63    cpu
                          13    1.01   0.96    gpu
                          39    2.81   2.40    gpu
                          118  13.04   6.79    gpu
                 dtde     5     1.14   1.24    cpu
                          13    2.47   2.73    cpu
                          39    7.74   7.55    gpu
                          118  21.48  25.10    cpu
          paper  ctce     5     0.92   1.12    cpu
                          13    0.92   1.16    cpu
                          39    1.30   1.35    cpu
                          118   2.68   1.82    gpu
                 ctde     5     1.43   1.94    cpu
                          13    2.70   3.54    cpu
                          39    7.59   8.91    cpu
                          118  34.31  22.75    gpu
                 dtde     5     2.66   4.45    cpu
                          13    7.03  10.97    cpu
                          39   21.19  28.69    cpu
                          118  68.95  88.57    cpu

## e. With the market: hours per million environment steps

The RL layer's time (the faster device) plus the market's time per step, from
`../notebooks/benchmark_market_step.ipynb` on the market side
(`MARKET_STEP_CSV`). The market was timed on SimBench networks of 13, 39, 53, 93
and 118 agents; other N are interpolated (and held at the ends). Without that
file, `MARKET_MS` below can be filled in by hand.

This is one environment sampled in the driver process. RLlib can sample several
copies in parallel (`num_env_runners`), which divides the market's part by up to
the number of runners, memory allowing; not measured here.

In [7]:
MARKET_MS = None  # e.g. {13: 30, 39: 40, 118: 61}, if the CSV is not there

if MARKET_MS is None and MARKET_STEP_CSV.exists():
    market = pd.read_csv(MARKET_STEP_CSV)
    market = market[market.partner_preference == PARTNER_PREFERENCE]
    market = market.groupby("agents").total_ms.mean()
    MARKET_MS = {n: float(np.interp(n, market.index, market.values)) for n in AGENTS}
print("market ms per step:", MARKET_MS)

best = done.loc[done.groupby(["algorithm", "preset", "paradigm", "N"])
                ["h / 1M steps"].idxmin()]
estimate = best[["algorithm", "preset", "paradigm", "N", "device", "h / 1M steps"]]
estimate = estimate.rename(columns={"h / 1M steps": "RL h / 1M"})
if MARKET_MS:
    estimate["market h / 1M"] = estimate.N.map(MARKET_MS) * 1e6 / 1000 / 3600
    estimate["total h / 1M"] = estimate["RL h / 1M"] + estimate["market h / 1M"]
    estimate["steps per night (10 h)"] = (10 / estimate["total h / 1M"] * 1e6).round(-3)
estimate.set_index(["algorithm", "preset", "paradigm", "N"])

market ms per step: {5: 29.51169372924293, 13: 29.51169372924293, 39: 40.09478126924174, 118: 61.352789615436144}


device  RL h / 1M  market h / 1M  total h / 1M  steps per night (10 h)
algorithm preset paradigm N                                                                         
ppo       light  ctce     5      cpu       0.42           8.20          8.62                1.16e+06
                          13     cpu       0.51           8.20          8.71                1.15e+06
                          39     cpu       0.70          11.14         11.84                8.45e+05
                          118    gpu       1.36          17.04         18.40                5.43e+05
                 ctde     5      cpu       0.60           8.20          8.79                1.14e+06
                          13     gpu       0.96           8.20          9.16                1.09e+06
                          39     gpu       2.40          11.14         13.54                7.39e+05
                          118    gpu       6.79          17.04         23.84                4.20e+05
                 dtde     5      cpu       1.14           8.20          9.34                1.07e+06
                          13     cpu       2.47           8.20         10.67                9.37e+05
                          39     gpu       7.55          11.14         18.69                5.35e+05
                          118    cpu      21.48          17.04         38.52                2.60e+05
          paper  ctce     5      cpu       0.92           8.20          9.12                1.10e+06
                          13     cpu       0.92           8.20          9.12                1.10e+06
                          39     cpu       1.30          11.14         12.44                8.04e+05
                          118    gpu       1.82          17.04         18.87                5.30e+05
                 ctde     5      cpu       1.43           8.20          9.63                1.04e+06
                          13     cpu       2.70           8.20         10.90                9.17e+05
                          39     cpu       7.59          11.14         18.73                5.34e+05
                          118    gpu      22.75          17.04         39.79                2.51e+05
                 dtde     5      cpu       2.66           8.20         10.86                9.21e+05
                          13     cpu       7.03           8.20         15.23                6.57e+05
                          39     cpu      21.19          11.14         32.32                3.09e+05
                          118    cpu      68.95          17.04         85.99                1.16e+05

## f. How to read it

- **What it does not say: how many steps a problem needs.** That is the real
  unknown, and it grows with N; only learning runs on the market answer it. This
  benchmark turns a number of steps into hours.
- **The cost of the PPO settings.** `"paper"` against `"light"` shows how much of
  the time is the 30 epochs over minibatches of 128.
- **Where the time goes.** Large `sampling` is RLlib's per-step overhead (it
  grows with the number of modules in DTDE); large `learning` is gradient steps,
  which the GPU or lighter settings help with.